# [실습] Context Engineering과 미들웨어

LLM 에이전트는 컨텍스트가 길어질수록 응답 품질이 떨어지고 비용이 늘어납니다.
Context Engineering은 필요한 정보를 필요한 시점에만 LLM에 전달해 이 문제를 다루는 기술입니다.

LangChain 1.0의 미들웨어는 에이전트 실행 사이클의 각 단계에 개입하는 표준 수단입니다.
모델 호출 전후와 도구 실행 전후에 개입해 컨텍스트를 제어합니다.
이번 실습에서는 기본 제공 미들웨어를 사용하고, 커스텀 미들웨어를 직접 만듭니다.

![middleware](https://mintcdn.com/langchain-5e9cc07a/RAP6mjwE5G00xYsA/oss/images/middleware_final.png?fit=max&auto=format&n=RAP6mjwE5G00xYsA&q=85&s=eb4404b137edec6f6f0c8ccb8323eaf1)

## 학습 목표

- 컨텍스트 엔지니어링의 네 가지 전략(write, select, compress, isolate) 구분
- 미들웨어의 훅 구조와 ModelRequest, Runtime 객체 확인
- 기본 제공 미들웨어로 컨텍스트 압축(Summarization, ContextEditing)과 도구 선택(LLMToolSelector) 실행
- PIIMiddleware로 입력과 도구 결과의 개인정보 마스킹
- 커스텀 미들웨어 구현: 동적 컨텍스트 주입, 동적 모델 선택, 출력 가드레일, 호출 제한
- 여러 미들웨어를 하나의 파이프라인으로 조합

## 1. 환경 설정

In [ ]:
%pip install langchain langchain-openai langchain-tavily langchain-google-genai langgraph slack_sdk -q

In [ ]:
import os
import re
import time
import base64
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Any
from zoneinfo import ZoneInfo
from dotenv import load_dotenv

load_dotenv('.env', override=True)

# Slack SDK 감지
try:
    from slack_sdk import WebClient
    from slack_sdk.errors import SlackApiError
    _SLACK_SDK_INSTALLED = True
except ImportError:
    _SLACK_SDK_INSTALLED = False

_SLACK_AVAILABLE = (
    _SLACK_SDK_INSTALLED
    and bool(os.getenv("SLACK_BOT_TOKEN"))
    and bool(os.getenv("SLACK_CHANNEL_ID"))
)

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentMiddleware,
    SummarizationMiddleware,
    PIIMiddleware,
    hook_config,
)
from langchain.tools import tool
from langchain.messages import SystemMessage, AIMessage
from langchain_core.callbacks import UsageMetadataCallbackHandler
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver

print('환경 설정 완료')
print(f'Slack 모드: {"Live (실제 Slack 연동)" if _SLACK_AVAILABLE else "Mock (PII 포함 시뮬레이션 데이터)"}')

In [ ]:
from select_model import load_model

light_model = load_model(platform='vllm', enable_thinking=False, max_tokens=16384, temperature=0.7, top_p=0.80, presence_penalty=1.5, extra_body={"top_k": 20, "min_p": 0.0, "repetition_penalty": 1.0})
# 간단한 작업은 Light 모델을 사용

heavy_model = load_model(platform='vllm', enable_thinking=True,  max_tokens=16384, temperature=1.0, top_p=0.95, presence_penalty=1.5, extra_body={"top_k": 20, "min_p": 0.0, "repetition_penalty": 1.0})

print('light_model:', light_model.model_name, '| heavy_model:', heavy_model.model_name)

## 도구-미들웨어 매핑

도구별로 미들웨어 검증에 사용할 시나리오를 연결합니다.

### 도구 구성

| # | 도구명 | 설명 |
|---|--------|------|
| 1 | `tavily_search` | Tavily 웹 검색으로 최신 정보 조회 |
| 2 | `slack_get_messages` | Slack 채널 최근 메시지 조회 (Mock 모드 지원) |
| 3 | `slack_post_message` | Slack 채널에 메시지 전송 (Mock 모드 지원) |
| 4 | `fetch_url` | 특정 URL 내용 가져오기 |
| 5 | `save_memo` | 팀 메모 저장 (in-memory) |
| 6 | `get_memos` | 저장된 메모 조회 |

### 도구-미들웨어 매핑

| 미들웨어 | 대상 도구 | 시나리오 |
|----------|----------|--------|
| PIIMiddleware | `slack_get_messages` | Slack 메시지의 이메일/전화번호를 감지해 redact/mask |
| ContextInjection | `tavily_search` + `save_memo` | 사용자 부서/역할에 맞춘 검색 및 메모 |
| OutputGuardrail | `tavily_search` | 금융/의료 검색 결과에 면책 조항 자동 추가 |
| CallLimit | `fetch_url` + `tavily_search` | 다단계 작업에서 비용 높은 호출 제한 |
| Summarization | 전체 도구 (멀티턴) | 긴 도구 출력(Slack, 검색) 누적 시 토큰 관리 |

### Slack Mock 모드 안내

`.env` 파일에 `SLACK_BOT_TOKEN`과 `SLACK_CHANNEL_ID`가 있으면 실제 Slack API를 호출합니다.
미설정 시 PII(이메일, 전화번호)와 금융/의료 키워드가 포함된 Mock 데이터를 반환합니다.

---
## 2. 도구 정의


In [ ]:
# Slack 클라이언트 설정
_slack_client = None
_SLACK_CHANNEL = None
if _SLACK_AVAILABLE:
    _slack_client = WebClient(token=os.environ["SLACK_BOT_TOKEN"])
    _SLACK_CHANNEL = os.environ["SLACK_CHANNEL_ID"]

# Mock Slack 데이터
MOCK_SLACK_MESSAGES = [
    {"user": "김영희", "text": "내일 AI Transformation 팀 회의 참석 부탁드립니다. kim@company.com 로 zoom 링크 주소 보냈습니다.", "ts": "2026-03-03 09:15"},
    {"user": "박민수", "text": "고객 이메일 정리 완료: client1@example.com, client2@test.kr . 계약서 검토 바랍니다.", "ts": "2026-03-03 09:30"},
    {"user": "이지은", "text": "5월 2일 'LLM 에이전트 하네스 엔지니어링' 세미나 등록했습니다. notolab.ceo@gmail.com, 010-1234-3921 으로 등록했어요.", "ts": "2026-03-03 10:00"},
    {"user": "최현우", "text": "건강검진 결과 공유: 혈압 약 처방받았습니다. 복용 스케줄은 lee@hospital.kr 로 문의하세요.", "ts": "2026-03-03 10:15"},
    {"user": "정수연", "text": "긴급: 파트너사 담당자 park@partner.co.kr , API Limit 관련하여 등급 상향이 필요하다고 합니다. 010-9876-5432 로 오늘 중 연락 필요합니다.", "ts": "2026-03-03 10:30"},
]


# 메모 저장소
_memo_store: dict[str, str] = {}


@tool
def tavily_search(
    query: str,
    topic: str = "general",
    max_results: int = 3,
    time_range: str | None = None,
) -> str:
    """Tavily 웹 검색으로 최신 정보를 조회합니다.

    Args:
        query: 검색 질의어
        topic: 검색 주제 ("general" | "news"). 기본값 "general"
        max_results: 반환할 검색 결과 수 (1~20). 기본값 3
        time_range: 검색 기간 필터 ("day" | "week" | "month" | "year" | None). 기본값 None(전체 기간)
    """
    search = TavilySearch(max_results=max_results, topic=topic)
    results = search.invoke(query)['results']
    context = ""
    for r in results:
        if isinstance(r, dict):
            title = r.get("title", "N/A")
            url = r.get("url", "")
            content = r.get('content','No Contents')
            context += f"- {title}\n  URL: {url}\n  {content}\n\n"
        else:
            context += str(r) + "\n"
    return context if context else "검색 결과가 없습니다."

@tool
def fetch_url(url: str, max_length: int = 10000) -> str:
    """웹 페이지의 텍스트를 가져옵니다.

    Args:
        url: 가져올 URL
        max_length: 반환할 최대 글자 수
    """
    try:
        import httpx
    except ImportError:
        return "[ERROR] httpx가 설치되지 않았습니다: pip install httpx"

    try:
        with httpx.Client(follow_redirects=True, timeout=30) as client:
            resp = client.get(url, headers={"User-Agent": "LangChain-Agent/1.0"})
            resp.raise_for_status()

        content_type = resp.headers.get("content-type", "")
        if "text/html" in content_type:
            # HTML 본문을 텍스트로 변환
            from html.parser import HTMLParser

            class _TextExtractor(HTMLParser):
                def __init__(self):
                    super().__init__()
                    self.parts: list[str] = []
                    self._skip = False

                def handle_starttag(self, tag, attrs):
                    self._skip = tag in ("script", "style", "noscript")

                def handle_endtag(self, tag):
                    if tag in ("script", "style", "noscript"):
                        self._skip = False

                def handle_data(self, data):
                    if not self._skip:
                        text = data.strip()
                        if text:
                            self.parts.append(text)

            extractor = _TextExtractor()
            extractor.feed(resp.text)
            text = "\n".join(extractor.parts)
        else:
            text = resp.text

        if len(text) > max_length:
            text = text[:max_length] + f"\n... [truncated at {max_length} chars]"
        return f"[{resp.status_code}] {url}\n\n{text}"
    except Exception as e:
        return f"[ERROR] {type(e).__name__}: {e}"



@tool
def slack_get_messages(channel: str = "general", hours: int = 24) -> str:
    """Slack 채널의 최근 메시지를 조회합니다.
    Args:
        channel: 조회할 채널 이름 (기본값: general)
        hours: 조회할 시간 범위 (기본값: 24시간)
    """
    if _SLACK_AVAILABLE:
        try:
            oldest_ts = str(
                (datetime.now(timezone.utc) - timedelta(hours=hours)).timestamp()
            )
            resp = _slack_client.conversations_history(
                channel=_SLACK_CHANNEL, oldest=oldest_ts, limit=20
            )
            messages = resp.get("messages", [])
            if not messages:
                return f"최근 {hours}시간 동안 #{channel}에 메시지가 없습니다."
            lines = []
            for msg in sorted(messages, key=lambda m: float(m.get("ts", "0"))):
                text = msg.get("text", "").strip()
                user = msg.get("user", "unknown")
                if text:
                    lines.append(f"- {user}: {text}")
            return f"#{channel} 최근 {hours}시간 메시지 ({len(lines)}개):\n" + "\n".join(lines)
        except Exception as e:
            return f"Slack 메시지 조회 실패: {e}"
    else:
        # Mock 모드 데이터 반환
        lines = []
        for msg in MOCK_SLACK_MESSAGES:
            lines.append(f"[{msg['ts']}] {msg['user']}: {msg['text']}")
        return (
            f"#{channel} 최근 메시지 ({len(MOCK_SLACK_MESSAGES)} 개):\n"
            + "\n".join(lines)
        )


@tool
def slack_post_message(channel: str, message: str) -> str:
    """Slack 채널에 메시지를 전송합니다.

    Args:
        channel: 전송할 채널 이름
        message: 전송할 메시지 내용
    """
    if _SLACK_AVAILABLE:
        try:
            resp = _slack_client.chat_postMessage(
                channel=_SLACK_CHANNEL, text=message
            )
            return f"메시지가 #{channel}에 전송되었습니다. (ts: {resp.get('ts', '')})"
        except Exception as e:
            return f"메시지 전송 실패: {e}"
    else:
        print(f"  [Mock Slack] #{channel}에 메시지 전송: {message[:80]}...")
        return f"[Mock] 메시지가 #{channel}에 전송되었습니다."


@tool
def save_memo(title: str, content: str) -> str:
    """팀 메모를 저장합니다.

    Args:
        title: 메모 제목
        content: 메모 내용
    """
    _memo_store[title] = content
    memo_list = ", ".join(_memo_store.keys())
    return f"메모 '{title}'이(가) 저장되었습니다. 현재 저장된 메모: [{memo_list}]"


@tool
def get_memos() -> str:
    """저장된 모든 메모를 조회합니다."""
    if not _memo_store:
        return "저장된 메모가 없습니다."
    result = "저장된 메모 목록:\n"
    for title, content in _memo_store.items():
        result += f"  - [{title}]: {content}\n"
    return result.strip()


tools = [tavily_search, fetch_url, slack_get_messages, slack_post_message, save_memo, get_memos]
print(f"도구 {len(tools)}개 정의 완료: {[t.name for t in tools]}")

---
## 3. 기본 Agent

In [ ]:
basic_agent = create_agent(
    light_model,
    tools=tools,
    system_prompt="주어진 도구를 사용하여 질문에 답변하세요. 매 도구를 사용하기 전 중간 상황을 간단히 설명하세요.",
)

In [ ]:
_memo_store.clear()

test_questions = [
    "Slack 채널의 최근 메시지를 확인하고 요약해줘, 주요 연락처와 이메일 주소를 표로 출력해",
]

for q in test_questions:
    cb = UsageMetadataCallbackHandler()
    result = basic_agent.invoke(
        {"messages": [{"role": "user", "content": q}]},
        config={"callbacks": [cb]},
    )
    answer = result["messages"][-1].text
    print(f'Q: {q}')
    print(f'A: {answer}')
    print()

---
## 4. 컨텍스트 전략 네 가지

컨텍스트 엔지니어링은 한정된 어텐션 예산을 관리하는 일입니다.
토큰이 늘면 윈도우 안에 들어가더라도 응답 품질이 떨어질 수 있습니다.
컨텍스트에는 현재 요청에 필요한 정보만 넣습니다.

전략은 네 가지로 나뉩니다.

| 전략 | 핵심 | 예시 |
|------|------|------|
| write | 컨텍스트 밖에 기록 | 메모리, 메모 저장 |
| select | 필요한 것만 불러오기 | 도구 선택, 검색 |
| compress | 같은 정보를 더 짧게 | 요약, 오래된 도구 출력 정리 |
| isolate | 나눠서 처리 | 서브에이전트 |

이 모듈은 미들웨어로 select와 compress를 다룹니다.
도구 스키마 오버헤드는 선택으로, 도구 출력 누적은 압축으로 줄입니다.

---
## 5. 미들웨어 동작 원리

미들웨어는 `AgentMiddleware`를 상속해, 개입하려는 단계의 훅(Hook)만 골라 구현합니다. 각 훅은 에이전트 실행 사이클의 특정 지점에서 호출됩니다.

### 훅의 종류

| 훅 | 호출 시점 | 주 용도 |
|----|----------|---------|
| `before_agent` / `after_agent` | 에이전트 실행 전후 1회 | 초기화, 정리 |
| `before_model` / `after_model` | 매 모델 호출 전후 | 상태 점검, 응답 후처리 |
| `wrap_model_call` | 모델 호출을 감쌈 | 요청 수정(프롬프트, 도구, 모델) |
| `wrap_tool_call` | 도구 실행을 감쌈 | 도구 입출력 가공 |

- `before_*` / `after_*` 훅은 상태 업데이트 dict 또는 `None`을 반환합니다.
- `before_model`에 `@hook_config(can_jump_to=["end"])`를 붙이면 루프를 끊고 종료할 수 있습니다.
- `wrap_*` 훅은 요청을 받아 `handler(request)`를 호출하고 그 응답을 돌려줍니다. 호출 전후에 원하는 동작을 삽입합니다.
- 비동기 실행이 필요하면 a* 훅을 함께 구현합니다.

```python
class MyMiddleware(AgentMiddleware):
    def before_model(self, state, runtime):
        return None              # 상태 업데이트 dict 또는 None

    def wrap_model_call(self, request, handler):
        # request.override(...)로 요청 일부를 교체
        return handler(request)
```

---
## 6. 기본 제공 미들웨어

LangChain은 자주 쓰는 컨텍스트 제어 패턴을 미들웨어로 제공합니다.

### 6-1. SummarizationMiddleware (compress)

장기 대화에서 메시지가 누적되면 컨텍스트 윈도우를 초과할 수 있습니다.
`SummarizationMiddleware`는 지정한 조건에 도달하면 오래된 메시지를 요약해 토큰을 줄입니다.

| 파라미터 | 설명 |
|----------|------|
| `model` | 요약에 쓸 경량 모델 (비용 절감) |
| `trigger` | 요약 실행 조건: `("tokens", N)`, `("messages", N)`, `("fraction", 0.8)` |
| `keep` | 유지할 최근 메시지: `("messages", N)`, `("tokens", N)`, `("fraction", 0.3)` |

In [ ]:
_memo_store.clear()

# 요약 미들웨어 에이전트
summarized_agent = create_agent(
    light_model,
    tools=tools,
    system_prompt="당신은 유능한 AI 어시스턴트입니다. 한국어로 답변하세요.",
    middleware=[
        SummarizationMiddleware(
            model=light_model,        # 요약용 경량 모델
            trigger=("messages", 10),     # 메시지 10개 이상이면 요약
            keep=("messages", 5),         # 최근 5개 메시지는 유지
        ),
    ],
    checkpointer=InMemorySaver(),
)

# 비교용 기본 에이전트
baseline_agent_with_memory = create_agent(
    light_model,
    tools=tools,
    system_prompt="당신은 유능한 AI 어시스턴트입니다. 한국어로 답변하세요.",
    checkpointer=InMemorySaver(),
)

print('에이전트 생성 완료 (요약 vs 기본)')

In [ ]:
long_conversation = [
    "Slack 채널의 최근 메시지를 확인해줘",
    "세미나 주제가 뭐지?",
    "그 세미나 일정하고 제목만 메모에 저장해줘.",
    "세미나 주제에 대한 최근 동향을 링크 3개 정도로 소개해줄래?",
    "파트너는 왜 긴급 연락하라고 한거야? '내용이랑 같이 파트너사 대응' 메모로 저장해줘",
    "이번 대화에서 저장한 메모를 모두 보여주고, 오늘 논의한 내용을 한 문단으로 요약해줘",
]

thread_summ = {"configurable": {"thread_id": "summ-test"}}
thread_base = {"configurable": {"thread_id": "base-test"}}

summ_tokens = []
base_tokens = []

for i, msg in enumerate(long_conversation, 1):
    # 요약 에이전트
    cb1 = UsageMetadataCallbackHandler()
    r1 = summarized_agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config={**thread_summ, "callbacks": [cb1]},
    )
    t1 = sum(v.get("input_tokens", 0) for v in cb1.usage_metadata.values()) if cb1.usage_metadata else 0
    summ_tokens.append(t1)

    # 기본 에이전트
    cb2 = UsageMetadataCallbackHandler()
    r2 = baseline_agent_with_memory.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config={**thread_base, "callbacks": [cb2]},
    )
    t2 = sum(v.get("input_tokens", 0) for v in cb2.usage_metadata.values()) if cb2.usage_metadata else 0
    base_tokens.append(t2)

    msg_count_summ = len(r1["messages"])
    msg_count_base = len(r2["messages"])
    print(f'  턴 {i}: "{msg}"')
    print(f'    내용: {r1['messages'][-1].text[:800]} (중략)')
    print('========\n'*2)
    print(f'    요약 에이전트: 메시지 {msg_count_summ}개, 입력 토큰 {t1}')
    print(f'    기본 에이전트: 메시지 {msg_count_base}개, 입력 토큰 {t2}')

### 6-2. ContextEditingMiddleware (compress)

도구 출력이 길어지면 컨텍스트가 빠르게 증가합니다.
`ContextEditingMiddleware`는 입력 토큰이 임계치를 넘으면 오래된 도구 실행 결과를 제거해 윈도우를 확보합니다.

Summarization이 메시지 전체를 요약한다면, ContextEditing은 오래된 도구 출력을 제거합니다.
오래된 도구 출력 누적을 줄이는 흐름을 확인합니다.

In [ ]:
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit

edited_agent = create_agent(
    light_model,
    tools=tools,
    system_prompt="당신은 유능한 AI 어시스턴트입니다. 한국어로 답변하세요.",
    middleware=[
        # 짧은 입력에서도 정리가 실행되도록 trigger를 낮게 설정합니다.
        ContextEditingMiddleware(edits=[ClearToolUsesEdit(trigger=4000, keep=2)]),
    ],
)
print('ContextEditingMiddleware 에이전트 생성 완료')

### 6-3. LLMToolSelectorMiddleware (select)

도구가 많으면 스키마만으로 매 요청의 고정 오버헤드가 커집니다.

`LLMToolSelectorMiddleware`는 본 모델 호출 전에 독립된 LLM에게 질문하여 현 상황에서 필요한 도구를 필터링합니다.
`max_tools`로 선택할 도구 개수 상한을 둡니다.

In [ ]:
from langchain.agents.middleware import LLMToolSelectorMiddleware

selected_agent = create_agent(
    light_model,
    tools=tools,
    system_prompt="당신은 유능한 AI 어시스턴트입니다. 한국어로 답변하세요.",
    middleware=[LLMToolSelectorMiddleware(model=light_model, max_tools=3)],
)

r = selected_agent.invoke(
    {"messages": [{"role": "user", "content": "Slack 채널 메시지를 확인해줘"}]},
)
print(r["messages"][-1].text)

### 6-4. PIIMiddleware (보안)

`PIIMiddleware`는 사용자 입력이나 에이전트 출력에서 개인정보(PII)를 감지해 처리합니다.

| 전략 | 설명 | 예시 |
|------|------|------|
| `"redact"` | 완전 삭제 | `test@email.com` -> `[REDACTED]` |
| `"mask"` | 부분 마스킹 | `010-1234-5678` -> `010-xxxx-xxxx` |
| `"block"` | 발견 시 실행 차단 | 에러 반환 |
| `"hash"` | 해시값으로 대체 | 비가역적 치환 |

내장 PII 타입(`"email"`, `"credit_card"` 등) 외에 `detector` 파라미터로 커스텀 정규식을 추가할 수 있습니다.

In [ ]:
class PIILoggingMiddleware(AgentMiddleware):
    """LLM에 전달되는 메시지에서 PII 필터링 결과를 확인합니다.
    """

    def _log(self, request):
        for msg in request.messages:
            content = msg.text if hasattr(msg, 'text') else str(msg.content)
            if msg.type == 'human':
                print(f"  [PII 필터링 후 LLM 입력] {content}")

    def wrap_model_call(self, request, handler):
        self._log(request)
        return handler(request)

    async def awrap_model_call(self, request, handler):
        self._log(request)
        return await handler(request)


pii_agent = create_agent(
    light_model,
    tools=tools,
    system_prompt="당신은 고객센터 AI 어시스턴트입니다. 한국어로 친절하게 답변하세요.",
    middleware=[
        PIIMiddleware("email", strategy="redact",
                      apply_to_input=True, # HumanMessage에 적용
                      apply_to_tool_results = True # ToolMessage에 적용
                      ),
        PIIMiddleware(
            "phone_number",
            detector=r"01[016789]-?\d{3,4}-?\d{4}",
            strategy="mask",
            apply_to_input=True,
            apply_to_tool_results = True
        ),
        PIILoggingMiddleware(),  # PII 필터링 결과 출력
    ],
)

print('PIIMiddleware + PIILoggingMiddleware 에이전트 생성 완료')

In [ ]:
pii_tests = [
    "전화번호는 010-1234-5678입니다. 제 전화번호가 뭐라구요?",
    "환불 문의합니다. 연락처 010-9876-5432, 이메일 customer@notolab.net 입니다.",
]

for test_input in pii_tests:
    print(f'  [원본 입력] {test_input}')
    result = pii_agent.invoke(
        {"messages": [{"role": "user", "content": test_input}]},
    )
    print(f'  [에이전트 응답] {result["messages"][-1].text}...')
    print()

In [ ]:
# Slack 메시지 조회 시나리오
print('  --- Slack 메시지 PII 시나리오 ---')
print('  [원본 입력] Slack 채널의 최근 메시지 중 주요 연락처를 확인해줘')
r_slack = pii_agent.invoke(
    {"messages": [{"role": "user", "content": "Slack 채널의 최근 메시지 중 주요 연락처를 확인해줘"}]},
)
print(f'  [에이전트 응답] {r_slack["messages"][-1].text}')

---
## 7. 커스텀 미들웨어 만들기

`AgentMiddleware`를 상속하여 직접 미들웨어를 구성합니다.



### 핵심 객체: ModelRequest와 Runtime

`wrap_model_call`이 받는 `request`와 훅이 받는 `runtime`이 미들웨어의 핵심 객체입니다.

- ModelRequest: 모델 호출 요청입니다. 이번 실습에서는 `model`, `messages`, `system_message`를 확인하고 `request.override(...)`로 필요한 필드만 바꿉니다.
- Runtime: 훅 호출 시 전달되는 실행 스코프입니다.

### 여러 미들웨어의 실행 순서

| 훅 종류 | 순서 |
|---------|------|
| `before_*` | 등록한 순서대로 |
| `after_*` | 역순으로 |
| `wrap_*` | 첫 미들웨어가 가장 바깥, 마지막이 모델에 가장 가까움 |

### 7-1. ContextInjectionMiddleware (wrap_model_call)

`create_agent()`의 시스템 프롬프트는 생성 시점에 고정됩니다.
현재 시각이나 사용자 프로필처럼 런타임에 달라지는 정보는 호출 시점에 주입해야 합니다.

`wrap_model_call`로 매 LLM 호출 직전에 시스템 프롬프트에 동적 컨텍스트를 추가합니다.
다음 셀에서는 `request.override(system_message=...)`로 일부만 바꾼 새 요청을 만들어 `handler`에 넘깁니다.

In [ ]:
class ContextInjectionMiddleware(AgentMiddleware):
    """시스템 프롬프트에 현재 시간과 사용자 프로필을 주입합니다.
    """

    def __init__(self, user_profile: dict | None = None):
        super().__init__()
        self.user_profile = user_profile or {}

    def _inject(self, request):
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        profile_str = ", ".join(f"{k}: {v}" for k, v in self.user_profile.items())

        injection = f"\n\n[동적 컨텍스트]\n현재 시각: {now}"
        if profile_str:
            injection += f"\n사용자 프로필: {profile_str}"

        original = request.system_prompt or ""
        if "[동적 컨텍스트]" not in original:
            new_sys = SystemMessage(content=original + injection)
            request = request.override(system_message=new_sys)
            print(f'  [ContextInjection] 동적 컨텍스트 주입 완료')
        return request

    def wrap_model_call(self, request, handler):
        return handler(self._inject(request))

    async def awrap_model_call(self, request, handler):
        return await handler(self._inject(request))


print('ContextInjectionMiddleware 정의 완료')

In [ ]:
_memo_store.clear()

ctx_agent = create_agent(
    light_model,
    tools=tools,
    system_prompt="당신은 유능한 AI 어시스턴트입니다. 한국어로 답변하세요.",
    middleware=[
        ContextInjectionMiddleware(
            user_profile={"이름": "Justin Kim", "부서": "AI연구팀", "역할": "시니어 엔지니어",
            "연차":'10', 'Personal_Preference':'LLM Jailbreaking 연구중, 이모지를 아주 좋아함'}
        ),
    ],
)

# 시간 주입 확인
r = ctx_agent.invoke(
    {"messages": [{"role": "user", "content": "지금 몇 시야?"}]},
)
print(f'  응답: {r["messages"][-1].text[:100]}')
print()

# 프로필 반영과 검색 도구 활용 확인
r = ctx_agent.invoke(
    {"messages": [{"role": "user", "content": "안녕 GPT! 내 연구 주제에 맞는 최신 논문 소개해줄래?"}]},
)
print(f'  응답: {r["messages"][-1].text}')

### 7-2. OutputGuardrailMiddleware (after_model)

`after_model`에서 LLM 응답을 검증하고, 필요하면 면책 문구를 덧붙입니다.

- 금융/투자 응답: 투자 면책 조항 추가
- 의료/건강 응답: 전문가 상담 권고 추가
- 도구 호출 중간 단계: 검증 생략

금융/의료 주제 응답에서 면책 문구 추가 여부를 확인합니다.

In [ ]:
from langchain.messages import HumanMessage

class OutputGuardrailMiddleware(AgentMiddleware):
    """after_model에서 응답을 검증하고 면책 문구를 덧붙입니다.
    """

    def __init__(self):
        super().__init__()
        self.classifier = light_model
        self.classification_prompt = """다음 텍스트를 분류하세요. 해당하는 카테고리를 쉼표로 구분하여 반환하세요.
카테고리: finance, medical, none

분류 기준:
- finance: 투자, 주식, 수익률, 펀드, 배당, 매수, 매도, 코인, 환율 등 금융/투자 관련 내용
- medical: 약, 처방, 증상, 진단, 치료, 복용 등 의료/건강 관련 내용
- none: 위 카테고리에 해당하지 않는 경우

반드시 카테고리 이름만 반환하세요. 예: finance 또는 medical 또는 none

텍스트:
{text}"""

    def _parse(self, text):
        return [c.strip() for c in text.strip().lower().split(",")]

    def _classify(self, content):
        prompt = self.classification_prompt.format(text=content[:500])
        return self._parse(self.classifier.invoke([HumanMessage(content=prompt)]).text)

    async def _aclassify(self, content):
        prompt = self.classification_prompt.format(text=content[:500])
        resp = await self.classifier.ainvoke([HumanMessage(content=prompt)])
        return self._parse(resp.text)

    def _disclaimers(self, categories):
        out = []
        if "finance" in categories:
            out.append("[안내] 위 내용은 일반적인 정보이며, 투자 판단의 근거로 사용할 수 없습니다. 투자 결정은 전문가와 상담하세요.")
            print("  [Guardrail] 금융 면책 조항 추가")
        if "medical" in categories:
            out.append("[안내] 위 내용은 참고용이며, 정확한 진단 및 치료는 의료 전문가와 상담하세요.")
            print("  [Guardrail] 의료 면책 조항 추가")
        return out

    def _should_skip(self, state):
        last = state["messages"][-1]
        if getattr(last, "tool_calls", None):
            return True
        return not (last.text if hasattr(last, "text") else "")

    def _build_update(self, state, categories):
        last = state["messages"][-1]
        disclaimers = self._disclaimers(categories)
        if not disclaimers:
            return None
        updated = last.text + "\n\n" + "\n".join(disclaimers)
        # 같은 id의 메시지를 반환해 기존 응답을 교체합니다.
        return {"messages": [AIMessage(id=last.id, content=updated)]}

    def after_model(self, state, runtime):
        if self._should_skip(state):
            return None
        return self._build_update(state, self._classify(state["messages"][-1].text))

    async def aafter_model(self, state, runtime):
        if self._should_skip(state):
            return None
        return self._build_update(state, await self._aclassify(state["messages"][-1].text))


print("OutputGuardrailMiddleware 정의 완료")

In [ ]:
guardrail_agent = create_agent(
    light_model,
    tools=tools,
    system_prompt="당신은 유능한 AI 어시스턴트입니다. 한국어로 답변하세요.",
    middleware=[OutputGuardrailMiddleware()],
)

# 금융 응답 가드레일 확인
r1 = guardrail_agent.invoke(
    {"messages": [{"role": "user", "content": "AI가 금융 상품을 추천하는 서비스도 있니?"}]},
)
print(f'  금융 검색 응답:\n  {r1["messages"][-1].text}')
print()

# 의료 응답 가드레일 확인
r2 = guardrail_agent.invoke(
    {"messages": [{"role": "user", "content": "고혈압 치료 방법을 검색해줘"}]},
)
print(f'  의료 검색 응답:\n  {r2["messages"][-1].text}')

### 7-3. CallLimitMiddleware (before_model + can_jump_to)

에이전트가 무한 루프에 빠지거나 모델을 과도하게 호출하는 것을 막습니다.
이 미들웨어는 `before_model`에서 호출 횟수를 카운트합니다.

호출 횟수가 임계값을 넘으면 `@hook_config(can_jump_to=["end"])` 권한으로 `jump_to: "end"`를 반환해 에이전트를 종료합니다.

"검색 + 메모 저장 + Slack 조회" 같은 다단계 작업에서 호출 제한이 발동하는 것을 확인합니다.

In [ ]:
class CallLimitMiddleware(AgentMiddleware):
    """모델 호출 횟수를 제한하여 무한 루프를 방지합니다.
    """

    def __init__(self, max_calls: int = 5):
        super().__init__()
        self.max_calls = max_calls
        self.call_count = 0

    def _tick(self):
        """호출 횟수를 늘리고, 한도 초과 시 종료 응답을 반환합니다."""
        self.call_count += 1
        print(f'  [CallLimit] 모델 호출 {self.call_count}/{self.max_calls}')
        if self.call_count > self.max_calls:
            print(f'  [CallLimit] 최대 호출 횟수 초과 -> 강제 종료')
            return {
                "messages": [{
                    "role": "assistant",
                    "content": f"처리 중 모델 호출이 {self.max_calls}회를 초과하여 중단합니다. 질문을 나누어 다시 시도해주세요."
                }],
                "jump_to": "end"
            }
        return None

    @hook_config(can_jump_to=["end"])
    def before_model(self, state, runtime):
        return self._tick()

    @hook_config(can_jump_to=["end"])
    async def abefore_model(self, state, runtime):
        return self._tick()

    def after_agent(self, state, runtime):
        print(f'  [CallLimit] 최종 모델 호출 횟수: {self.call_count}')
        self.call_count = 0
        return None

    async def aafter_agent(self, state, runtime):
        return self.after_agent(state, runtime)


# 정상 범위 호출
_memo_store.clear()
limited_agent = create_agent(
    light_model,
    tools=tools,
    system_prompt="당신은 유능한 AI 어시스턴트입니다. 한국어로 답변하세요.",
    middleware=[CallLimitMiddleware(max_calls=5)],
)

r = limited_agent.invoke(
    {"messages": [{"role": "user", "content": "Slack 채널 메시지를 확인하고 요약해줘"}]},
)
print(f'  응답: {r["messages"][-1].text[:100]}')
print()

# 다단계 요청에서 호출 제한 확인
print('  --- 다단계 요청 (호출 제한 테스트) ---')
_memo_store.clear()
limited_agent2 = create_agent(
    light_model,
    tools=tools,
    system_prompt=(
        "당신은 유능한 AI 어시스턴트입니다. 한국어로 답변하세요. "
        "도구는 반드시 하나씩 순서대로 호출하세요. 절대 여러 도구를 동시에 호출하지 마세요."
    ),
    middleware=[CallLimitMiddleware(max_calls=3)],
)

r2 = limited_agent2.invoke(
    {"messages": [{"role": "user", "content": (
        "다음을 순서대로 처리해줘: "
        "1) LangGraph 최신 정보 검색, "
        "2) 검색 결과를 '기술 동향' 메모로 저장, "
        "3) Slack 채널 메시지 조회, "
        "4) Slack 내용을 '팀 현황' 메모로 저장"
    )}]},
)
print(f'  응답: {r2["messages"][-1].text[:100]}')

### 7-4. ModelRouterMiddleware (동적 모델 선택)

`wrap_model_call`에서 요청을 보고 `request.override(model=...)`로 모델을 바꿉니다.
짧은 호출은 `light_model`, 긴 호출은 `heavy_model`로 보냅니다.

앞에서 불러온 `light_model`과 `heavy_model`을 라우팅 대상으로 사용합니다.
이 셀에서는 입력 길이를 라우팅 기준으로 사용합니다.

In [ ]:
class ModelRouterMiddleware(AgentMiddleware):
    """입력 길이에 따라 easy 또는 hard 모델을 선택합니다."""

    def __init__(self, easy, hard, hard_chars=2000):
        super().__init__()
        self.easy, self.hard, self.hard_chars = easy, hard, hard_chars

    def _pick(self, request):
        user = next((m for m in reversed(request.messages) if m.type == "human"), None)
        text = user.text if user else ""
        model = self.hard if len(text) > self.hard_chars else self.easy
        print(f'  [ModelRouter] {"pro" if model is self.hard else "open"} 선택 (입력 {len(text)}자)')
        return request.override(model=model)

    def wrap_model_call(self, request, handler):
        return handler(self._pick(request))

    async def awrap_model_call(self, request, handler):
        return await handler(self._pick(request))


routed_agent = create_agent(
    light_model,
    tools=tools,
    system_prompt="당신은 유능한 AI 어시스턴트입니다. 한국어로 답변하세요.",
    middleware=[ModelRouterMiddleware(easy=light_model, hard=heavy_model)],
)
print("ModelRouterMiddleware 에이전트 생성 완료")

---
## 8. 미들웨어 조합

여러 미들웨어를 `middleware=[...]` 리스트로 조합할 수 있습니다.
등록 순서와 훅 종류에 따라 실행 순서가 달라집니다.

```
입력 - PIIMiddleware (PII 마스킹)
     - ContextInjection (시간/프로필 주입)
     - [LLM 호출]
     - OutputGuardrail (면책 조항 추가)
     - CallLimit (호출 횟수 체크)
     - SummarizationMiddleware (장기 대화 요약)
```

In [ ]:
_memo_store.clear()

full_agent = create_agent(
    light_model,
    tools=tools,
    system_prompt="당신은 유능한 AI 어시스턴트입니다. 한국어로 답변하세요.",
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware(
            "phone_number",
            detector=r"01[016789]-?\d{3,4}-?\d{4}",
            strategy="mask",
            apply_to_input=True,
        ),
        ContextInjectionMiddleware(user_profile={"이름": "김그림", "부서": "개발팀", "연차": "1", "History":"구글의 이미지 생성 모델 Nano Banana 2를 활용하는 어플리케이션 개발중"}),
        OutputGuardrailMiddleware(),
        CallLimitMiddleware(max_calls=5),
        SummarizationMiddleware(
            model=light_model,
            trigger=("messages", 5),
            keep=("messages", 4),
        ),
    ],
    checkpointer=InMemorySaver(),
)

r = full_agent.invoke(
    {"messages": [{"role": "user", "content": "내가 쓰는 모델에 도움될 프롬프트 가이드 있나? 원본 링크도 줘"}]},
    config={"configurable": {"thread_id": "full-test"}},
)
print(f'  응답: {r["messages"][-1].text}')

## build_agent로 배포 에이전트 만들기

배포 팩토리 build_agent.py는 middleware 인자로 노트북에서 만든 미들웨어를 받습니다. ContextInjectionMiddleware를 넘기면 매 모델 호출 직전에 시스템 프롬프트에 현재 시각과 사용자 프로필이 주입된 채로 배포 에이전트가 동작합니다.

In [ ]:
%%writefile build_agent.py
"""build_agent.py

에이전트를 한 곳에서 조립하는 팩토리입니다. 모델은 select_model로 고르고,
도구, MCP, 미들웨어를 추가한 버전입니다.
MCP 도구를 await로 받아오므로 build_agent는 async 함수입니다.
"""

from __future__ import annotations

import os

from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

from select_model import load_model
from tools import DEFAULT_TOOLS
from mcp_config import load_servers

DEFAULT_SYSTEM_PROMPT = (
    '당신은 유용한 AI 어시스턴트입니다. '
    '필요하면 도구를 사용하고, 도구를 실행하기 전 중간 과정을 간략히 설명하세요.'
)

# build_agent가 직접 연결하는 MCP 서버. mcp_servers.json의 stateless 서버를 읽는다.
# stateful 세션(Playwright 등)은 봇이 세션으로 열어 extra_tools로 넘긴다.
MCP_SERVERS = load_servers(stateful=False)


async def _auto_elicitation(mcp_context, params, context):
    """Slack MCP의 post_message가 채널을 되물을 때 기본 채널로 자동 응답합니다.

    봇이나 노트북에는 사용자에게 입력 폼을 표시할 방법이 없으므로, SLACK_CHANNEL_ID
    환경변수의 채널로 답합니다.
    """
    from mcp.types import ElicitResult

    channel = os.environ.get('SLACK_CHANNEL_ID', 'C_DRYRUN')
    return ElicitResult(action='accept', content={'channel_id': channel})


async def load_mcp_tools(servers=None):
    """MCP 서버에서 도구를 받아옵니다. 연결에 실패하면 빈 목록을 돌려줍니다."""
    servers = MCP_SERVERS if servers is None else servers
    if not servers:
        return []
    try:
        from langchain_mcp_adapters.callbacks import Callbacks

        client = MultiServerMCPClient(
            servers, callbacks=Callbacks(on_elicitation=_auto_elicitation)
        )
        return await client.get_tools()
    except Exception as e:
        print(f'[build_agent] MCP 도구 로드 실패, 표준 도구만 사용합니다: {e}')
        return []


async def build_agent(
    model_name=None,
    platform='openai',
    model=None,
    tools=None,
    extra_tools=None,
    middleware=None,
    system_prompt=None,
    checkpointer=None,
    mcp_servers=None,
    **model_kwargs,
):
    """현재까지 구성된 에이전트를 만들어 반환합니다.

    Args:
        model_name: 모델 이름. select_model.load_model로 전달됩니다.
        platform: 'openai' | 'vllm' | 'ollama'. load_model로 전달됩니다.
        model: 이미 만든 chat model. 주면 model_name/platform은 무시됩니다.
        tools: 표준 도구 대신 쓸 도구 목록. 생략 시 tools.py의 DEFAULT_TOOLS.
        extra_tools: 호출자가 직접 만든 도구를 추가로 붙입니다. 봇이 stateful
            세션(예: Playwright)에서 받은 도구를 넘길 때 씁니다.
        middleware: create_agent에 넘길 미들웨어 목록. 노트북에서 만든
            미들웨어를 넣으면 그 동작이 배포 에이전트에 적용됩니다.
        system_prompt: 시스템 프롬프트. 생략 시 기본값을 씁니다.
        checkpointer: 멀티턴 대화를 위한 체크포인터.
        mcp_servers: 연결할 MCP 서버 설정. 생략 시 MCP_SERVERS, {}면 MCP를 끕니다.
        model_kwargs: load_model로 전달되는 추가 인자 (reasoning_effort 등).
    """
    if model is None:
        model = load_model(model_name, platform=platform, **model_kwargs)
    base_tools = list(DEFAULT_TOOLS) if tools is None else list(tools)
    mcp_tools = await load_mcp_tools(mcp_servers)
    extra = list(extra_tools) if extra_tools else []
    if system_prompt is None:
        system_prompt = DEFAULT_SYSTEM_PROMPT
    return create_agent(
        model,
        tools=base_tools + mcp_tools + extra,
        middleware=list(middleware) if middleware else [],
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )

build_agent.py를 import해 ContextInjectionMiddleware를 단 에이전트를 만듭니다. light_model을 직접 넘기고 `mcp_servers={}`로 tools.py의 표준 도구만 사용합니다. 시스템 프롬프트에 현재 시각과 사용자 프로필이 주입됩니다.

In [ ]:
from build_agent import build_agent

agent = await build_agent(
    model=light_model,
    middleware=[
        ContextInjectionMiddleware(
            user_profile={"이름": "Justin Kim", "부서": "AI연구팀", "역할": "시니어 엔지니어"}
        ),
    ],
    mcp_servers={},
)

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "지금 몇 시야?"}]},
)
print(result["messages"][-1].text)

## 정리

- 컨텍스트 엔지니어링 전략: write, select, compress, isolate로 한정된 어텐션 예산 관리
- 미들웨어 동작 원리: `AgentMiddleware`의 훅과 ModelRequest, Runtime 객체로 실행 사이클에 개입
- 기본 제공 미들웨어: Summarization과 ContextEditing으로 압축, LLMToolSelector로 도구 선택, PIIMiddleware로 개인정보 마스킹
- 커스텀 미들웨어: ContextInjection, OutputGuardrail, CallLimit, ModelRouter 구현
- 조합: 여러 미들웨어를 리스트로 합쳐 하나의 파이프라인 구성